In [ ]:
%load_ext autoreload
%autoreload 2
from hypnose_behavior.metric_analysis.movement.speed_analysis import run_speed_analysis_batch
from hypnose_behavior.utils.helpers import print_cache_keys
from hypnose_behavior.visualization.movement.speed import (
    plot_epoch_speeds_by_condition,
    plot_traces_with_speed_threshold,
)
from hypnose_behavior.visualization.movement.summary_stats import plot_movement_analysis_statistics
from hypnose_behavior.visualization.movement.tortuosity import plot_tortuosity_lines_overlay
from hypnose_behavior.visualization.movement.traces import (
    plot_movement_with_behavior,
    plot_trial_traces_by_mode,
)
from hypnose_behavior.io.save import use_style
%matplotlib widget

use_style('presentation')


## Plot movement traces for different modes - plot_trial_trace_by_mode quick guide

- Common args: subjid, dates (list [] or range ()), xlim, ylim, smooth_window (frames), linewidth, alpha, invert_y
- Modes: 
    - rewarded: rewarded trials only
    - rewarded_hr: rewarded trials; HR trials colored with HR palette
    - completed: all completed trials (rewarded, unrewarded, timeout)
    - all_trials: completed and aborted trials
    - fa_by_response: FA trials (selected by fa_types filter), sorted by response port
    - fa_by_odor: FA trials, sorted by each aborted odor
    - hr_only: hidden-rule trials, colored by associated reward port, with rewarded and unrewarded trials
- Options: 
    - show_average: adds mean trace + SEM per category
    - highlight_hr: in rewarded/all_trials mode, recolor HR trials in different palette
    - color_by_index: ignore categories; color each trace by normalized sample index
    - fa_types: filter FA labels (select between "FA_time_in", "FA_time_out", or both)

In [ ]:
plot_trial_traces_by_mode(
    subjid=45,
    dates=[20260107],
    mode='completed',
    xlim=(170, 970),
    ylim=(110, 950),
    position_units='cm',
    arena_size_cm=50.0,
    show_average=False, 
    highlight_hr=True, 
    color_by_index=False,
    color_by_speed=False,
    color_by_trial_id=False,
    fa_types=['FA_time_in'],
    figsize=(10, 6),
    save=True, 
    show_title=False,
    show_legend=False
)

# Modes: 
    # rewarded, rewarded_hr, completed, all_trials, fa_by_response, fa_by_odor, hr_only

# Speed Analysis

In [ ]:
speed_analysis = run_speed_analysis_batch(
    subjids=[57, 58, 59],
    dates=[20260717],
    fa_label_filter=["fa_time_in", "fa_time_out"],
    threshold=True
)

In [ ]:
speed_analysis_plot = plot_epoch_speeds_by_condition(
    subjid=59,
    dates=[20260717],
    fa_label_filter=["fa_time_in", "fa_time_out"],
    save=True
)


In [ ]:
fig_thresh = plot_traces_with_speed_threshold(
    subjid=57,
    dates=[20260717],
    position_units='cm',
    arena_size_cm=50.0,
    fa_types=["FA_time_in"],
    pre_buffer_s=0.5,
    threshold_alpha=10.0,
    threshold_beta=10.0,
    smooth_window=5,
    invert_y=True, 
    save=True
)

In [ ]:
figs_overlay = plot_tortuosity_lines_overlay(
    subjid=45,
    dates=[20260107],
    bin_ms=100,
    fixed_start_xy=(595, 135),
    fixed_goal_a_xy=(215, 940),
    fixed_goal_b_xy=(953, 950),
    save=True
)

In [ ]:
onset_latency = plot_movement_analysis_statistics(
    subjid=40,
    dates=[20251223],
    fa_types=["FA_time_in"],
    clean_graph=False,
    hidden_rule_analysis=True,
    save=True
)

# Functionalities

In [ ]:
plt.close('all')

In [ ]:
print_cache_keys()

# Miscellaneous

In [ ]:
# modes can be simple (all movement), trial_state (within trial vs outside), last_odor (A vs B), trial_windows (one or more trial windows), time_windows (one or more time windows), or trial_windows_rew
# for trial_windows: trial_windows=[(0, 20), (-20, None)] will plot first vs last 20 trials
# for time_windows: time_windows=[("15:20:00","15:25:00"), ("16:00:00","16:05:00")] will plot 2 5-minute windows
plot_movement_with_behavior(40, 20251211, mode='time_windows', time_windows=[("14:42:12", "14:42:55")],trial_windows=[(0, 10), (-10,None)], xlim=(100,1160), ylim=(10,950))

In [ ]:
# Cell to load single video and extract a single frame with ROI (saved in input folder for publications). 
# currently saves as tiff files, and using hardcoded input file.

import cv2
import os

video_path = "/Volumes/harris/hypnose/rawdata/sub-040_id-259/ses-046_date-20251229/behav/2025-12-29T16-21-37/VideoData/VideoData_1904-01-10T21-00-00.avi"
output_folder = video_path.rsplit("/", 1)[0]  # Save in the same folder as the video

#time_selection:
time_window = ["00:04:34.400"]

roi_x1, roi_y1 = 405, 10
roi_x2, roi_y2 = 685, 290
#ROIs for square around poke port: 
#roi_x1, roi_y1 = 405, 10
#roi_x2, roi_y2 = 785, 390
# ==========================

def parse_time_to_seconds(t_str):
    parts = t_str.split(":")
    if len(parts) == 3:
        h, m, s = parts
    elif len(parts) == 2:
        h = "0"
        m, s = parts
    else:
        raise ValueError(f"Unrecognized time format: {t_str}")
    return int(h) * 3600 + int(m) * 60 + float(s)

cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise IOError(f"Cannot open video: {video_path}")

fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

video_name = os.path.splitext(os.path.basename(video_path))[0]

for t_str in time_window:
    target_time_sec = parse_time_to_seconds(t_str)
    target_frame_idx = round(target_time_sec * fps)
    target_frame_idx = max(0, min(target_frame_idx, total_frames - 1))

    cap.set(cv2.CAP_PROP_POS_FRAMES, target_frame_idx)
    ret, frame = cap.read()
    if not ret:
        print(f"Failed to read frame for time {t_str} (frame {target_frame_idx})")
        continue

    # Actual timestamp landed on (for filename/log)
    actual_time_sec = target_frame_idx / fps
    h = int(actual_time_sec // 3600)
    m = int((actual_time_sec % 3600) // 60)
    s = int(actual_time_sec % 60)
    ms = int(round((actual_time_sec - int(actual_time_sec)) * 1000))

    # Crop the ROI directly from the original frame (no drawing, no quality loss)
    roi_crop = frame[roi_y1:roi_y2, roi_x1:roi_x2]

    safe_t = t_str.replace(":", "-")
    out_name = f"{video_name}_frame{target_frame_idx}_{safe_t}_ROI.tiff"
    out_path = os.path.join(output_folder, out_name)

    cv2.imwrite(out_path, roi_crop, [cv2.IMWRITE_TIFF_COMPRESSION, 1])

    print(f"Requested: {t_str}  |  fps={fps:.3f}")
    print(f"Nearest frame: {target_frame_idx}  |  Actual time: {h:02d}:{m:02d}:{s:02d}.{ms:03d}")
    print(f"Saved cropped ROI: {out_path}")

cap.release()
